# CNU Campus ChatBot — Colab 실행 노트북
**박연진 | 자연어처리 텀프로젝트**

## 실행 순서
```
[최초 1회]  셀 A → 런타임 재시작 → 셀 B → 셀 0 이후 순서대로
[이후 세션] 셀 0 (진입점) → 셀 3~7
```

| 셀 | 목적 | 실행 조건 |
|:---:|---|---|
| A | 패키지 설치 | **최초 1회만** → 런타임 재시작 |
| B | 최초 clone | **최초 1회만** |
| 0 | 진입점 (Drive + 경로 + 패키지 확인) | **매 세션마다** |
| 1 | 데이터 갱신 (TTL 크롤링) | 선택적 |
| 2 | ChromaDB 구축 | chroma_db 없을 때만 |
| 3 | 7B 모델 Drive 캐시 확인/저장 | 최초 또는 Drive 삭제 시 |
| 4 | 실행 전 체크리스트 | 선택적 |
| 5 | chatbot.sh (추론 + UI) | **핵심 실행** |
| 6 | chat_output.json 확인 | 검증용 |
| 7 | UI만 단독 실행 | 재실행/데모 |


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 A] 패키지 설치 — 최초 1회만 실행
# 완료 후 [런타임] > [런타임 다시 시작] 클릭
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q \
    "transformers>=4.40.0" \
    accelerate \
    bitsandbytes \
    datasets \
    scikit-learn \
    kiwipiepy \
    rank_bm25 \
    chromadb \
    sentence-transformers \
    gradio \
    requests \
    beautifulsoup4 \
    tqdm

print("설치 완료!")
print("→ [런타임] > [런타임 다시 시작] 클릭 후 셀 B 실행")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 B] 레포 Clone — 최초 1회만 실행
# 이미 clone 됐으면 건너뛰고 셀 0 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

PROJECT = '/content/drive/MyDrive/NLP_Term_Project'

if os.path.exists(PROJECT):
    print(f"이미 존재: {PROJECT}")
    print("→ Clone 건너뜀. 셀 0 실행으로 이동하세요.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive
    !git clone https://github.com/capyjin/NLP_Term_Project.git
    print("Clone 완료!")
    !ls NLP_Term_Project/

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 0] 진입점 — 매 세션마다 가장 먼저 실행
# Drive 마운트 + 경로 설정 + git pull + 패키지 확인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os, sys, importlib
PROJECT = '/content/drive/MyDrive/NLP_Term_Project'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("작업 경로:", os.getcwd())

# GPU 확인
import torch
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {gb:.1f} GB")
else:
    print("CPU 환경 — [런타임] > [런타임 유형 변경] > T4 GPU 선택 필요")

# 최신 코드 Pull
!git pull origin main

# 패키지 확인
print()
for pkg in ['transformers', 'bitsandbytes', 'kiwipiepy', 'rank_bm25', 'chromadb', 'gradio', 'sentence_transformers']:
    try:
        importlib.import_module(pkg)
        print(f"  OK {pkg}")
    except ImportError:
        print(f"  !! {pkg} 없음 -> 셀 A 실행 후 런타임 재시작 필요")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 1] 데이터 갱신 — 식단/셔틀/공지 크롤링
# TTL: 식단 6h / 셔틀 24h / 공지 6h
# 이미 최신이면 건너뛰어도 됨
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!python scripts/refresh_data.py
# 특정 항목만 갱신하려면:
# !python scripts/refresh_data.py --meal      # 식단만
# !python scripts/refresh_data.py --shuttle   # 셔틀만
# !python scripts/refresh_data.py --notice    # 공지/장학만

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 2] ChromaDB 구축 — chroma_db 없을 때만 실행
# 있으면 건너뜀 (Drive에 영구 저장됨)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, shutil, json

# FAQ inject 상태 확인
chunks_path = 'data/processed/chunks.json'
if os.path.exists(chunks_path):
    with open(chunks_path, encoding='utf-8') as f:
        chunks = json.load(f)
    faq_count = sum(1 for c in chunks if c.get('source_type') == 'faq_manual')
    print(f"chunks.json: {len(chunks)}개 (FAQ: {faq_count}개)")
    if faq_count < 23:
        print("FAQ 부족 -> inject 실행")
        !python scripts/inject_faq.py
    else:
        print("FAQ OK")

# chroma_db 구축
if os.path.exists('chroma_db') and len(os.listdir('chroma_db')) > 0:
    print("chroma_db 이미 존재 -> 건너뜀")
    print("  (재구축 필요 시: shutil.rmtree('chroma_db') 후 재실행)")
else:
    print("chroma_db 구축 시작...")
    !python src/vectordb/build_db.py
    if os.path.exists('chroma_db'):
        print("chroma_db 구축 완료!")
    else:
        print("구축 실패 - 위 오류 확인 필요")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 3] Qwen2.5-7B Drive 캐시 확인/저장
# Drive에 없으면 HuggingFace에서 다운로드 (~15GB, 20~30분)
# 이미 저장됐으면 즉시 완료
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DRIVE_7B = '/content/drive/MyDrive/models/qwen2.5-7b-4bit'
HF_7B    = 'Qwen/Qwen2.5-7B-Instruct'

if os.path.exists(DRIVE_7B) and os.path.exists(os.path.join(DRIVE_7B, 'config.json')):
    print(f"7B 모델 Drive 캐시 확인: {DRIVE_7B}")
    print("  -> 이미 저장됨. chatbot.sh에서 Drive 경로로 빠르게 로드됩니다.")
else:
    print(f"Drive 캐시 없음 -> {HF_7B} 다운로드 시작 (~15GB, 20~30분)")
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model = AutoModelForCausalLM.from_pretrained(
        HF_7B,
        quantization_config=quant,
        device_map='auto',
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(HF_7B)
    os.makedirs(DRIVE_7B, exist_ok=True)
    model.save_pretrained(DRIVE_7B)
    tokenizer.save_pretrained(DRIVE_7B)
    print(f"Drive 저장 완료: {DRIVE_7B}")
    del model  # VRAM 해제
    import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 4] 실행 전 체크리스트
# 모든 항목이 OK여야 chatbot.sh 정상 실행 가능
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json

checks = {
    "chroma_db 구축됨": os.path.exists('chroma_db') and len(os.listdir('chroma_db')) > 0,
    "chunks.json 존재": os.path.exists('data/processed/chunks.json'),
    "meal_menu.json 존재": os.path.exists('data/raw/meal_menu.json'),
    "shuttle_bus.json 존재": os.path.exists('data/raw/shuttle_bus.json'),
    "test_chat.json 존재": os.path.exists('data/test_chat.json'),
    "7B Drive 캐시 존재": os.path.exists('/content/drive/MyDrive/models/qwen2.5-7b-4bit/config.json'),
    "chatbot.sh 존재": os.path.exists('chatbot.sh'),
    "src/chatbot_ui.py 존재": os.path.exists('src/chatbot_ui.py'),
}

all_ok = True
for name, ok in checks.items():
    status = 'OK' if ok else '!!'
    print(f"  [{status}] {name}")
    if not ok: all_ok = False

# test_chat.json 없으면 placeholder 생성
if not os.path.exists('data/test_chat.json'):
    placeholder = [
        {"id": 1, "question": "졸업 학점이 몇 점인가요?"},
        {"id": 2, "question": "장학금 신청은 어디서 해?"},
        {"id": 3, "question": "수강신청 변경 기간은 언제야?"},
        {"id": 4, "question": "오늘 학식 뭐 나와?"},
        {"id": 5, "question": "셔틀버스 시간표 알려줘"},
    ]
    with open('data/test_chat.json', 'w', encoding='utf-8') as f:
        json.dump(placeholder, f, ensure_ascii=False, indent=2)
    print("  -> test_chat.json placeholder 생성 완료")

print()
if all_ok:
    print("모든 준비 완료 -> 셀 5 (chatbot.sh) 실행하세요!")
else:
    print("!! 위 항목 확인 후 해당 셀 재실행 필요")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 5] chatbot.sh 실행
# 전체 파이프라인: 크롤 -> 추론 -> UI
#   [3/4] test_chat.json -> outputs/chat_output.json 생성
#   [4/4] Gradio UI 실행 -> public URL 출력
# Drive 캐시 있으면 ~30초, 없으면 ~30분 소요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!bash chatbot.sh 2>&1

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 6] chat_output.json 결과 확인
# chatbot.sh 완료 후 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, os

path = 'outputs/chat_output.json'
if os.path.exists(path):
    with open(path, encoding='utf-8') as f:
        result = json.load(f)
    print(f"chat_output.json 생성 완료 - {len(result)}건\n")
    for i, item in enumerate(result, 1):
        q = item.get('question', item.get('user', ''))
        a = item.get('answer', item.get('model', ''))
        src = item.get('source', '')
        print(f"Q{i}: {q}")
        print(f"A{i}: {a[:120]}..." if len(a) > 120 else f"A{i}: {a}")
        if src:
            print(f"   [경로: {src}]")
        print()
else:
    print("chat_output.json 없음 - chatbot.sh 실행 중 오류 확인")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 7] UI 단독 실행 (데모/재실행용)
# chatbot.sh 없이 UI만 띄울 때 사용
# 이미 chatbot.sh로 UI가 실행 중이면 건너뜀
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 옵션 1: 데이터 갱신 + UI
!bash run_ui.sh

# 옵션 2: 데이터 갱신 건너뛰고 UI만
# !bash run_ui.sh --skip

# 옵션 3: UI 직접 실행
# !python src/chatbot_ui.py